# R2-1 / R3: Decomposing external knowledge — retrieval vs generation channel

Run this **after Stage 1** (00 → 00b → 01 → 02) has regenerated `results/tables/generation_results.csv` with the new conditions. It reads the master table and computes the two causal channels and the metadata leave-one-out drops. (The conditions themselves are defined in notebook 01's `build_context` and run by notebook 02 — no builder code lives here.)

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv('results/tables/generation_results.csv')
have = set(df.condition.unique())
need = {'EV00','EV10','EV01','EV11','ABL_full','ABL_nograph','ABL_nogloss','ABL_noval','ABL_nojoin'}
missing = need - have
assert not missing, f'Run Stage 1 first; missing conditions: {sorted(missing)}'
df = df[df.dataset=='D2']  # BIRD: the evidence regime

## Evidence 2×2 — separate the recall channel from the direct generation channel
`EV10−EV00` = effect routed through retrieval recall; `EV01−EV00` = direct effect on generation; `EV11` = combined (the operating point). Super-additivity (EV11 > EV10+EV01−EV00) indicates interaction.

In [ ]:
def ex(cond, tier='STRONG'):
    s=df[(df.condition==cond)&(df.model_tier==tier)]
    return s.exec_match.mean(), s.col_recall_strict.mean()
for tier in ['STRONG','MID','WEAK']:
    e00,r00=ex('EV00',tier); e10,r10=ex('EV10',tier); e01,r01=ex('EV01',tier); e11,r11=ex('EV11',tier)
    print(f'[{tier}] EX: EV00={e00:.3f} EV10={e10:.3f} EV01={e01:.3f} EV11={e11:.3f}')
    print(f'        recall channel (EV10-EV00)={e10-e00:+.3f} [recall {r00:.3f}->{r10:.3f}] | '
          f'direct gen (EV01-EV00)={e01-e00:+.3f} | interaction={e11-(e10+e01-e00):+.3f}')

## Metadata leave-one-out — independent contribution of each component
Each `ABL_no*` removes exactly one component from `ABL_full`; the EX/recall drop is that component's contribution.

In [ ]:
rows=[]
for tier in ['STRONG','MID','WEAK']:
    base=ex('ABL_full',tier)[0]
    for comp,cond in [('graph traversal','ABL_nograph'),('business glossary','ABL_nogloss'),
                      ('representative values','ABL_noval'),('mined join patterns','ABL_nojoin')]:
        e,r=ex(cond,tier); rows.append({'tier':tier,'component':comp,'EX':round(e,3),'dEX_vs_full':round(e-base,3)})
abl=pd.DataFrame(rows); print(abl.to_string(index=False))
abl.to_csv('results/tables/NEW_metadata_ablation.csv',index=False)

**Interpretation:** if most of the evidence gain appears in `EV10−EV00` (recall channel) rather than `EV01−EV00` (direct), external knowledge helps chiefly by improving schema linking — the paper's mechanism. The ablation's `nogloss`/`nojoin` rows also quantify the cost of operating without curated metadata (Reviewer 3's generalization concern).